# TIDE Case Tutorial: Mouse Embryonic Brain (End-to-End)

This notebook demonstrates a complete TIDE workflow on one real case:
1. Load a velocity-ready AnnData (mouse embryonic brain)
2. Run TIDE transition-specific graph refinement
3. Visualize cell-type UMAP and velocity stream with benchmark-style aesthetics
4. Extract high-cos modules by cell type
5. Extract transition-specific modules for one source cell
6. Export results for downstream analysis


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import matplotlib as mpl
import matplotlib.pyplot as plt

from tide import (
    TIDEGraphParams,
    compute_velocity_graph,
    extract_high_cos_by_celltype,
    extract_cell_to_target_high_cos,
    export_cell_direction_modules,
)

# Benchmark-style plotting config
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.unicode_minus"] = False

DPI = 150
TITLE_FS = 16
AXIS_FS = 12
TICK_FS = 10
LEGEND_FS = 10

METHOD_COLORS = {
    "Baseline": "#E6864A",
    "CellDancer": "#52BCA3",
    "UniTVelo": "#6E83B7",
    "tiVelo": "#D77FB3",
}

CATEGORY_PALETTE = [
    "#E6864A", "#52BCA3", "#6E83B7", "#D77FB3",
    "#8E8E8E", "#B79A6B", "#7DA063", "#4C78A8",
]


In [ ]:
# Change paths if needed
h5ad_path = Path('/path/to/velocity_ready_mousebrain.h5ad')
outdir = Path('./tide_case_outputs_mouse_embryonic_brain')
outdir.mkdir(parents=True, exist_ok=True)

assert h5ad_path.exists(), f'Input file not found: {h5ad_path}'
print('Input:', h5ad_path)
print('Output dir:', outdir.resolve())


In [ ]:
adata = sc.read_h5ad(h5ad_path)
print(adata)
print('layers:', list(adata.layers.keys()))
print('obs cols (head):', list(adata.obs.columns)[:12])

required_layers = ['velocity', 'Ms', 'Mu', 'velocity_u']
missing = [k for k in required_layers if k not in adata.layers]
if missing:
    raise KeyError(f'Missing required layers: {missing}')

# Align with AAA主图皮层分叉.py: use celltype as primary cluster column
if 'celltype' in adata.obs:
    celltype_key = 'celltype'
elif 'clusters' in adata.obs:
    celltype_key = 'clusters'
else:
    raise KeyError('This tutorial expects adata.obs["celltype"] or adata.obs["clusters"].')
print('celltype_key =', celltype_key)


In [ ]:
params = TIDEGraphParams(
    vkey='velocity',
    xkey='Ms',
    graph_mode='pairwise2d_geneTopK',
    ukey='Mu',
    vkey_u='velocity_u',
    pairwise_topk_genes=100,
    pairwise_candidate_genes=300,
    pairwise_use_signed_cos=True,
    store_gene_stats=True,
    store_k=100,
    store_edge_topm=30,
    store_edge_weight='sim_pos',
    n_jobs=8,
    show_progress_bar=False,
)

compute_velocity_graph(adata, params=params)
print('Done: adata.uns["velocity_graph"] and adata.uns["velocity_gene_cosine_stats"] are available.')

In [ ]:
# UMAP and velocity visualization (benchmark-style)
scv.settings.set_figure_params(dpi=DPI, dpi_save=DPI, frameon=False, fontsize=TICK_FS)
# 1) Cell-type UMAP with benchmark-style categorical palette
cats = sorted(adata.obs[celltype_key].astype(str).unique().tolist())
palette = CATEGORY_PALETTE[: len(cats)]
if len(palette) < len(cats):
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    need = len(cats) - len(palette)
    extra = [mcolors.to_hex(cm.get_cmap("tab20")(i / max(need - 1, 1))) for i in range(need)]
    palette = palette + extra
fig, ax = plt.subplots(figsize=(6.6, 5.4), dpi=DPI)
sc.pl.umap(
    adata,
    color=celltype_key,
    palette=palette,
    ax=ax,
    show=False,
    frameon=False,
    title="Mouse embryonic brain | cell types",
)
ax.tick_params(axis="both", labelsize=TICK_FS)
plt.show()
# 2) Velocity stream colored by cell type (compat patch for scvelo<=0.2.5 + newer pandas)
import importlib
scv_plot_utils = importlib.import_module("scvelo.plotting.utils")
scv_scatter_mod = importlib.import_module("scvelo.plotting.scatter")
from matplotlib import patheffects
from scvelo.plotting.utils import get_groups
def _set_legend_compat(
    adata, ax, value_to_plot, legend_loc, scatter_array,
    legend_fontweight, legend_fontsize, legend_fontoutline, legend_align_text, groups,
):
    if legend_fontoutline is None:
        legend_fontoutline = 1
    obs_vals = adata.obs[value_to_plot]
    obs_vals = obs_vals.cat.rename_categories([str(x) for x in obs_vals.cat.categories])
    adata.obs[value_to_plot] = obs_vals
    color_keys = adata.uns[f"{value_to_plot}_colors"]
    if isinstance(color_keys, dict):
        color_keys = np.array([color_keys[c] for c in obs_vals.cat.categories])
    valid_cats = np.where(obs_vals.value_counts()[obs_vals.cat.categories] > 0)[0]
    categories = np.array(obs_vals.cat.categories)[valid_cats]
    colors = np.array(color_keys)[valid_cats]
    if groups is not None:
        groups, groupby = get_groups(adata, groups, value_to_plot)
        groups = [g for g in groups if g in categories]
        colors = [colors[list(categories).index(x)] for x in groups]
        categories = groups
    if legend_loc == "on data":
        legend_fontweight = "bold" if legend_fontweight is None else legend_fontweight
        texts = []
        for label in categories:
            x_pos, y_pos = np.nanmedian(scatter_array[obs_vals == label, :], axis=0)
            if isinstance(label, str):
                label = label.replace("_", " ")
            kwargs = dict(verticalalignment="center", horizontalalignment="center")
            kwargs.update(dict(weight=legend_fontweight, fontsize=legend_fontsize))
            pe = [patheffects.withStroke(linewidth=legend_fontoutline, foreground="w")]
            text = ax.text(x_pos, y_pos, label, path_effects=pe, **kwargs)
            texts.append(text)
    else:
        for idx, label in enumerate(categories):
            if isinstance(label, str):
                label = label.replace("_", " ")
            ax.scatter([], [], c=[colors[idx]], label=label)
        ncol = 1 if len(categories) <= 14 else 2 if len(categories) <= 30 else 3
        kwargs = dict(frameon=False, fontsize=legend_fontsize, ncol=ncol)
        if legend_loc == "upper right":
            ax.legend(loc="upper left", bbox_to_anchor=(1, 1), **kwargs)
        elif legend_loc == "lower right":
            ax.legend(loc="lower left", bbox_to_anchor=(1, 0), **kwargs)
        elif "right" in legend_loc:
            ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), **kwargs)
        elif legend_loc != "none":
            ax.legend(loc=legend_loc, **kwargs)
scv_plot_utils.set_legend = _set_legend_compat
scv_scatter_mod.set_legend = _set_legend_compat
adata.obs[celltype_key] = adata.obs[celltype_key].astype("category")
adata.obs[celltype_key] = adata.obs[celltype_key].cat.rename_categories([str(x) for x in adata.obs[celltype_key].cat.categories])
scv.tl.velocity_embedding(adata, vkey="velocity", basis="umap")
scv.pl.velocity_embedding_stream(
    adata,
    vkey="velocity",
    basis="umap",
    color=celltype_key,
    palette=palette,
    legend_loc="right margin",
    dpi=DPI,
    save=False,
)


In [ ]:
# 1) High-cos modules by cell type
by_type = extract_high_cos_by_celltype(
    adata,
    celltype_key=celltype_key,
    vkey='velocity',
    top_n=50,
)

by_type_path = outdir / 'high_cos_genes_by_type_top50.csv'
by_type.to_csv(by_type_path, index=False)
print('saved:', by_type_path)
by_type.head()


In [ ]:
# 2) One-cell directional modules for case-study branch: Subplate -> Deeper/Upper
# Select a Subplate cell with strongest local bifurcation evidence in 30NN
if "distances" not in adata.obsp:
    sc.pp.neighbors(adata, n_neighbors=30)

labels = adata.obs[celltype_key].astype(str).values
is_subplate = labels == "Subplate"
is_upper = labels == "Upper Layer"
is_deeper = labels == "Deeper Layer"

D = adata.obsp["distances"].tocsr()
cands = np.where(is_subplate)[0]
rows = []
for i in cands:
    neigh = D.indices[D.indptr[i]:D.indptr[i+1]]
    neigh = neigh[neigh != i]
    n_upper = int(np.sum(is_upper[neigh]))
    n_deeper = int(np.sum(is_deeper[neigh]))
    score = min(n_upper, n_deeper)
    rows.append((i, n_upper, n_deeper, score))

branchable = [r for r in rows if r[1] > 0 and r[2] > 0]
if len(branchable) > 0:
    branchable = sorted(branchable, key=lambda x: (x[3], x[1] + x[2]), reverse=True)
    source_cell_idx, nU, nD, scv = branchable[0]
else:
    source_cell_idx, nU, nD, scv = rows[0][0], rows[0][1], rows[0][2], rows[0][3]

print("source_cell_idx =", int(source_cell_idx), ", celltype =", labels[int(source_cell_idx)])
print("local 30NN upper/deeper counts =", int(nU), int(nD))

target_types = ["Deeper Layer", "Upper Layer"]
pair_frames = []
for tgt in target_types:
    df_tgt = extract_cell_to_target_high_cos(
        adata,
        cell_idx=int(source_cell_idx),
        celltype_key=celltype_key,
        target_types=[tgt],
        vkey="velocity",
        top_n=50,
    )
    pair_frames.append(df_tgt)
    tgt_name = tgt.replace(" ", "_")
    tgt_path = outdir / f"cell{int(source_cell_idx)}_subplate_to_{tgt_name}_top50.csv"
    df_tgt.to_csv(tgt_path, index=False)
    print("saved:", tgt_path)

pair_df = pd.concat(pair_frames, axis=0, ignore_index=True)
combined_path = outdir / f"cell{int(source_cell_idx)}_subplate_to_deeper_upper_top50_combined.csv"
pair_df.to_csv(combined_path, index=False)
print("saved:", combined_path)
pair_df.head()


## What to do next

- Compare module outputs between methods (e.g., Baseline vs TIDE) using the same source cells and target directions.
- Perform ORA/GSEA on exported directional modules.
- Build embryonic-brain lineage-focused figures from the exported transition-specific modules.
